In [85]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [86]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [87]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [88]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [89]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.item_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lý null

### Xử lý age_group

In [90]:
df_age = read_parquet_item("./preprocessed-dataset")
df_age.head()

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final,age_group_from_desc_str,age_group_final,age_group_final_before
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cut (Trên 9 tháng) …","""Không xác định""","""9M+""","""Từ 9M""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định""","""Bé Gái""",null,"""Từ 36M""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm bằng chất liệu silicone mềm, dẻo và nước đã được chưng cất đảm bảo an toàn cho bé. - Dành…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Miếng Gặm Nướu Papa (CEQ004) (Cá hồng) Chất liệu: Silicone m…","""Không xác định""",null,"""0-12M""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miếng là sản phẩm dành cho bé 4-8kg đến từ thương hiệu uy tín Merries của Nhật Bản. Ra đời vớ…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""[""Từ 4M"", ""3M-6M"", ""12-36M""]""","""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M 58 miếng là sản phẩm dành cho bé từ 6-11kg đến từ thương hiệu uy tín Merries của Nhật Bản.…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""12-36M""","""12-36M"""


#### Ý tưởng 2 (tiếp theo):

##### Thống kê

In [91]:
import polars as pl

df = df_age   # hoặc df_filled tùy bạn đang dùng

# Hàm tiện dụng để lấy danh sách unique của 1 cột
def get_unique_list(df, col):
    return (
        df.select(col)
          .unique()
          .sort(col)
          .get_column(col)
          .to_list()
    )

# Lấy tất cả class
age_list    = get_unique_list(df, "age_group_final")

# Gom vào dictionary
category_dict = {
    "age_group": age_list
}

print(len(age_list))

# In ra theo từng dòng, rất dễ đọc
print("\n=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===\n")
for key, value in category_dict.items():
    print(f"{key}: {value}\n")

343

=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===

age_group: ['0-10M', '0-11M', '0-120M', '0-12M', '0-144M', '0-14M', '0-18M', '0-1M', '0-24M', '0-2M', '0-30M', '0-36M', '0-3M', '0-48M', '0-4M', '0-5M', '0-60M', '0-6M', '0-72M', '0-84M', '0-9M', '1-12M', '1-18M', '1-24M', '1-2M', '1-36M', '1-3M', '108-120M', '108-132M', '12-108M', '12-120M', '12-132M', '12-144M', '12-14M', '12-192M', '12-24M', '12-36M', '12-48M', '12-60M', '12-72M', '120-144M', '12M-18M', '12M-4Y', '13-16M', '13-17M', '132-144M', '13M-24M', '14-17M', '18-24M', '18M-24M', '18M-36M', '18M-4Y', '1D+', '1M-12M', '1M-15M', '1M-3M', '216-360M', '216-720M', '24-120M', '24-36M', '24-48M', '24-60M', '24-72M', '24-96M', '2M-15M', '2M-6M', '3-12M', '3-18M', '3-24M', '3-36M', '3-60M', '3-6M', '36-120M', '36-144M', '36-216M', '36-48M', '36-60M', '36-72M', '3M-12M', '3M-18M', '3M-24M', '3M-6M', '4-24M', '4-30M', '4-6M', '48-60M', '48-72M', '4M-4Y', '4M-6M', '5D+', '6-12M', '6-144M', '6-15M', '6-18M', '6-24M', '6-30M', '6-36M', 

In [92]:
import polars as pl
import textwrap

# ===============================
# 1) LỌC DỮ LIỆU CẦN XUẤT
# ===============================
import polars as pl

# Điều kiện "có thông tin" cho description
desc_has_info = (
    pl.col("description").is_not_null()
    & (pl.col("description") != "Không xác định")
    & (pl.col("description").str.strip_chars() != "")
)

# Điều kiện "có thông tin" cho description_new
desc_new_has_info = (
    pl.col("description_new").is_not_null()
    & (pl.col("description_new") != "Không xác định")
    & (pl.col("description_new").str.strip_chars() != "")
)

# Lọc các dòng cần thống kê
df_unknown_desc = (
    df_age
    .filter(
        (pl.col("age_group_final") == "Không xác định")
        & (desc_has_info | desc_new_has_info)
    )
    .select([
        "item_id",
        "description",
        "description_new",
        "age_group_final",
    ])
)
print("Số dòng cần xuất:", df_unknown_desc.height)

# ===============================
# 2) HÀM FORMAT TEXT
# ===============================
def wrap175(text):
    if text is None:
        return ""
    return textwrap.fill(str(text), width=175)

# ===============================
# 3) XUẤT RA FILE TXT
# ===============================
output_file = "unknown_description_items.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for row in df_unknown_desc.iter_rows(named=True):
        f.write("=====================================\n")
        f.write(f"item_id: {row['item_id']}\n\n")

        f.write("age_group_final: ")
        f.write(f"{row['age_group_final']}\n\n")

        f.write("description:\n")
        f.write(wrap175(row["description"]) + "\n\n")

        f.write("description_new:\n")
        f.write(wrap175(row["description_new"]) + "\n\n")

print(f"Đã xuất file: {output_file}")


Số dòng cần xuất: 3297
Đã xuất file: unknown_description_items.txt


In [93]:
# Thống kê
total_rows = df_age.height
unknown_after = df_age.filter(pl.col("age_group_final") == "Không xác định").height
ratio_after = unknown_after / total_rows * 100

print(f"Số dòng 'Không xác định': {unknown_after}")
print(f"Tỉ lệ: {ratio_after:.2f}%")
print()

Số dòng 'Không xác định': 10362
Tỉ lệ: 37.92%



##### Thử fill lại

Cấu hình & tiền xử lý

In [94]:
import re
from typing import Optional, List, Tuple

# Nếu bạn dùng polars cho pipeline phía ngoài thì giữ dòng này
import polars as pl

# =========================================
# 1. CẤU HÌNH CONTEXT & TỪ KHÓA
# =========================================

POS_CONTEXT = [
    # Trẻ em / baby
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
    "thiếu nhi", "newborn", "sơ sinh",

    # Ngữ cảnh độ tuổi
    "độ tuổi",
    "tháng tuổi",
    "months old",
    "years old",
    "tuổi",
]

CHILD_WORDS = [
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
]

# Đơn vị thời gian dùng để quy đổi -> THÁNG
MONTH_WORDS = ["tháng", "thang", "month", "months"]
YEAR_WORDS  = ["tuổi", "year", "years", "y"]
WEEK_WORDS  = ["tuần", "tuan", "week", "weeks"]
DAY_WORDS   = ["ngày", "ngay", "day", "days", "ngày tuổi", "ngay tuoi"]

WINDOW_POS = 80   # dùng để tìm POS context
WINDOW_NEG = 100  # dùng để tìm NEG context


# =========================================
# 2. PREPROCESS TEXT TRƯỚC KHI MATCH
# =========================================

def preprocess_text_for_age(text: Optional[str]) -> str:
    """
    Chuẩn hoá text:
      - Nếu None -> ""
      - Chuẩn hoá khoảng trắng
      - Chèn space chỗ dính chữ: 'hợpBé' -> 'hợp Bé'
      - Chèn space trước '(' nếu dính
      - Chèn space sau ':' nếu thiếu
      - Chuẩn hoá 'NB-12' -> '0-12' (NB = newborn)
    """
    if text is None:
        return ""

    t = str(text)

    # Chuẩn hoá khoảng trắng (bao gồm cả non-breaking space)
    t = re.sub(r"\s+", " ", t)

    # Tách giữa 1 ký tự bất kỳ (không phải space) và chữ hoa (kể cả Đ)
    # 'Độ tuổi phù hợpBé' -> 'Độ tuổi phù hợp Bé'
    t = re.sub(r"([^\s])([A-ZĐ])", r"\1 \2", t)

    # 'Formula(1-3 tuổi)' -> 'Formula (1-3 tuổi)'
    t = re.sub(r"([a-zA-ZÀ-ỹ])\(", r"\1 (", t)

    # 'Độ tuổi:5-7Y' -> 'Độ tuổi: 5-7Y'
    t = re.sub(r":([^\s])", r": \1", t)

    # NB-12 -> 0-12 (NB = newborn)
    t = re.sub(r"\bnb\s*-\s*(\d{1,2})", r"0-\1", t, flags=re.IGNORECASE)

    return t.strip()


In [95]:
# =========================================
# 3. REGEX PATTERNS CHO EXTRACT
# =========================================

# Range đặc biệt "3M-5T"
AGE_MT_RANGE_PATTERN = re.compile(
    r"(?P<n1>\d{1,2})\s*m\s*(?:-|–|—|đến|to)\s*(?P<n2>\d{1,2})\s*t"
)

# "0-6 tháng ... đến ... 3 tuổi"
CROSS_MONTH_RANGE_TO_YEAR_PATTERN = re.compile(
    r"(?P<m_start>\d{1,2})\s*-\s*(?P<m_end>\d{1,2})\s*"
    r"(?:tháng|thang|month|months)"
    r".{0,40}?(?:đến|tới|to)\s*.{0,40}?"
    r"(?P<y_end>\d{1,2}(?:[.,]\d+)?)\s*(?:tuổi|year|years|y)",
)

# "sơ sinh ... đến ... 3 tuổi"
NEWBORN_RANGE_PATTERN = re.compile(
    r"(?:từ\s+giai\s*đoạn\s+)?sơ\s*sinh"
    r".{0,30}?(?:đến|tới|to|-).{0,30}?"
    r"\d{1,2}(?:[.,]\d+)?\s*(tháng|thang|month|months|tuổi|year|years|y)",
)

# Newborn phrase không số: "trẻ sơ sinh trở lên", "từ trẻ sơ sinh"
NEWBORN_PHRASE_PATTERN = re.compile(
    r"(?:từ\s+)?(?:trẻ\s+)?sơ\s*sinh(?:\s*(trở lên|\+))?"
)

# "trong độ tuổi từ 1 đến 5", "độ tuổi từ 2 đến 6"
AGE_WORD_RANGE_PATTERN = re.compile(
    r"độ\s*tuổi[^0-9]{0,40}"
    r"(?:từ\s*)?"
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)"
)

# "6 tháng đến 5 tuổi", "từ 1 tuần - 1 tuổi"
RANGE_BOTH_UNITS_PATTERN = re.compile(
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u1>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
    r"(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u2>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
)

# "0-12 tháng", "từ 2-6 tuổi", "Từ 1-36 tháng tuổi"
RANGE_ONE_UNIT_PATTERN = re.compile(
    r"(?:(?:từ)\s*)?"
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
)

# Single: "dưới 12 tháng", "từ 6 tháng tuổi", "trên 6 tuổi",
#         "không quá 3 tuổi", "4 tháng tuổi", "từ 0 tháng tuổi trở lên"
SINGLE_PATTERN = re.compile(
    r"(?:(?P<cmp>dưới|under|<|từ|trên|hơn|sau|>=|over|more than|không quá|khong qua)\s*)?"
    r"(?P<n>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u>"
    r"tháng tuổi|tháng|thang|month|months|"
    r"tuổi|year|years|y|"
    r"tuần|tuan|week|weeks|"
    r"ngày tuổi|ngày|ngay|day|days"
    r")"
    r"(?:\s*(?P<suffix>trở lên|\+))?"
)


In [96]:
# =========================================
# 4. HÀM CONTEXT POS / NEG
# =========================================

def has_pos_context(context: str) -> bool:
    return any(tok in context for tok in POS_CONTEXT)


def has_neg_context(context: str) -> bool:
    """
    NEG = những câu rõ ràng là KHÔNG khuyến nghị cho trẻ,
    hoặc mô tả nguy cơ, hoặc dạng 'cần dùng dưới sự hướng dẫn của bác sĩ'.
    KHÔNG coi 'hạn sử dụng', 'bảo quản', 'bảo hành' là NEG nữa.
    """
    ctx = context

    # 1) Không phù hợp / không dùng cho trẻ
    if any(phrase in ctx for phrase in [
        "không thích hợp cho trẻ",
        "không thích hợp với trẻ",
        "không dùng cho trẻ",
        "không nên dùng cho trẻ",
        "không phù hợp cho trẻ",
        "không phù hợp với trẻ",
        "không sử dụng sản phẩm cho trẻ",
        "không sử dụng cho trẻ",
    ]):
        return True

    # 2) 'tránh dùng/sử dụng ... cho trẻ'
    if "tránh" in ctx and any(c in ctx for c in CHILD_WORDS):
        if any(v in ctx for v in ["dùng", "sử dụng", "cho"]):
            return True

    # 3) Nguy cơ / nguy hiểm / rủi ro + có 'trẻ'
    if any(word in ctx for word in ["nguy cơ", "nguy hiểm", "rủi ro"]):
        if any(c in ctx for c in CHILD_WORDS):
            return True

    # 4) English NEG
    if "not suitable for children" in ctx or "do not use for children" in ctx:
        return True

    # 5) 'cần sử dụng dưới sự hướng dẫn/giám sát của bác sĩ / chuyên gia'
    if ("dưới sự hướng dẫn" in ctx or "dưới sự giám sát" in ctx) and (
        "bác sĩ" in ctx or "chuyên gia" in ctx
    ):
        return True

    # 6) Các câu kiểu 'trong 6 tháng đầu đời' (thời gian, không phải tuổi)
    if "tháng đầu" in ctx or "tháng đầu đời" in ctx:
        if any(k in ctx for k in ["trong ", "trong vòng", "giai đoạn"]):
            return True

    return False


# =========================================
# 5. HELPER: NHẬN DIỆN PHRASE CÂN NẶNG
# =========================================

def is_weight_phrase(match_text: str, left_context: str, right_context: str) -> bool:
    """
    Trả True nếu cụm này nhiều khả năng nói về CÂN NẶNG, không phải TUỔI:
      - match_text chứa 'kg', 'kilogram', ...
      - HOẶC right_context gần bên phải chứa 'kg' (trường hợp dính liền 20-35kgBỉm)
      - HOẶC có 'cân nặng' / 'trẻ nặng' / 'bé nặng' trong match_text, left_context hoặc right_context.
    """
    s = match_text
    lc = left_context
    rc = right_context

    # 1) '... 10-15kg' nằm trong match_text
    if any(kw in s for kw in ["kg", "kilogram", "ký", "kí"]):
        return True

    # 2) '... 10-15kgBỉm' -> 'kg' nằm ngay sau match
    if any(kw in rc[:10] for kw in ["kg", "kilogram", "ký", "kí"]):
        return True

    # 3) 'cân nặng', 'trẻ nặng', 'bé nặng'
    weight_markers = ["cân nặng", "trẻ nặng", "bé nặng"]
    if any(kw in s for kw in weight_markers):
        return True
    if any(kw in lc for kw in weight_markers):
        return True
    if any(kw in rc for kw in weight_markers):
        return True

    return False


# =========================================
# 6. HELPER: NHẬN DIỆN DURATION "sau X tháng sử dụng"
# =========================================

# Các pattern mô tả THỜI LƯỢNG SỬ DỤNG, HẠN SỬ DỤNG, KHÔNG PHẢI TUỔI
DURATION_PATTERNS = [
    # "sau 3 tháng sử dụng", "sau 2 tuần dùng"
    re.compile(
        r"sau\s+\d{1,2}\s*"
        r"(tháng|thang|month|months|tuần|tuan|week|weeks)\s+"
        r"(sử dụng|su dung|dùng|dung)"
    ),

    # "sử dụng không quá 1 tháng", "sử dụng trong vòng 2 tháng"
    re.compile(
        r"(sử dụng|su dung|dùng|dung)\s+"
        r"(trong vòng|không quá)\s*"
        r"\d{1,2}\s*"
        r"(tháng|thang|month|months|tuần|tuan|week|weeks)"
    ),

    # "hạn sử dụng: 18 tháng", "hạn dùng: 6 tháng"
    re.compile(
        r"hạn\s*(sử dụng|dùng)?\s*:\s*"
        r"\d{1,2}\s*(tháng|thang|month|months)"
    ),

    # "hạn sử dụng 18 tháng", "hạn dùng 6 tháng"
    re.compile(
        r"hạn\s*(sử dụng|dùng)?\s+"
        r"\d{1,2}\s*(tháng|thang|month|months)"
    ),

    # "thời hạn sử dụng 18 tháng"
    re.compile(
        r"thời\s*hạn\s*(sử dụng|dùng)?\s*"
        r"\d{1,2}\s*(tháng|thang|month|months)"
    ),

    # "18 tháng kể từ ngày sản xuất"
    re.compile(
        r"\d{1,2}\s*(tháng|thang|month|months)\s+kể từ ngày sản xuất"
    ),

    # "1 tháng sau khi mở hộp/nắp/chai/lon"
    re.compile(
        r"\d{1,2}\s*(tháng|thang|month|months)\s+sau khi mở\s*(nắp|hộp|chai|lon)"
    ),

    # "sau khi sử dụng từ 1-2 tháng"
    re.compile(
        r"sau khi\s+(sử dụng|su dung|dùng|dung)\s+từ\s*"
        r"\d{1,2}\s*-\s*\d{1,2}\s*"
        r"(tháng|thang|month|months|tuần|tuan|week|weeks)"
    ),
]


def is_duration_context(match_text: str, context: str) -> bool:
    """
    Trả True nếu CỤM match_text (số + đơn vị) đang ở trong bối cảnh
    mô tả THỜI LƯỢNG sử dụng / HẠN SỬ DỤNG, không phải tuổi.

    - Nếu match_text chứa 'tuổi', 'year', 'y', 'tháng tuổi' -> coi là tuổi, KHÔNG phải duration.
    - Ngược lại, thử match các DURATION_PATTERNS lên context cục bộ quanh match.
    """
    # 1) Đơn vị tuổi -> chắc chắn là tuổi, không phải duration
    if any(tok in match_text for tok in ["tuổi", "year", "years", "y", "tháng tuổi"]):
        return False

    # 2) Duyệt các pattern duration
    for pat in DURATION_PATTERNS:
        if pat.search(context):
            return True

    return False


In [97]:
# =========================================
# 7. HÀM EXTRACT_AGE_PHRASES (DÙNG desc_all)
# =========================================

def extract_age_phrases(desc_all: Optional[str]) -> List[str]:
    """
    Nhận 1 chuỗi desc_all (merge 2 description),
    preprocess và trả về list các cụm tuổi/tháng/ngày.
    """
    if desc_all is None:
        return []

    pre = preprocess_text_for_age(desc_all)
    if not pre:
        return []

    text = pre.lower()

    matches: List[str] = []
    used_spans: List[Tuple[int, int]] = []

    def overlap_span(start: int, end: int) -> bool:
        return any(not (end <= s or start >= e) for (s, e) in used_spans)

    def get_contexts(start_idx: int, end_idx: int) -> Tuple[str, str, str, str]:
        left_pos = max(0, start_idx - WINDOW_POS)
        right_pos = min(len(text), end_idx + WINDOW_POS)
        context_pos = text[left_pos:right_pos]

        left_neg = max(0, start_idx - WINDOW_NEG)
        right_neg = min(len(text), end_idx + WINDOW_NEG)
        context_neg = text[left_neg:right_neg]

        left_weight = text[max(0, start_idx - 40):start_idx]
        right_weight = text[end_idx:min(len(text), end_idx + 40)]

        return context_pos, context_neg, left_weight, right_weight

    def handle_match(m: re.Match):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            return

        ctx_pos, ctx_neg, left_weight, right_weight = get_contexts(start_idx, end_idx)
        match_text = text[start_idx:end_idx].strip()

        # 1) Bỏ các cụm về CÂN NẶNG
        if is_weight_phrase(match_text, left_weight, right_weight):
            return

        # 2) Bỏ các cụm mô tả THỜI LƯỢNG SỬ DỤNG (sau X tháng sử dụng/dùng)
        local_ctx = text[max(0, start_idx - 60):min(len(text), end_idx + 60)]
        if is_duration_context(match_text, local_ctx):
            return

        # 3) Nếu đơn vị là TUẦN (week), yêu cầu phải có usage-keywords
        #    để tránh câu mô tả như "khi được 6 đến 8 tuần thì giấc ngủ..."
        if any(w in match_text for w in ["tuần", "tuan", "week", "weeks"]):
            if not any(kw in ctx_pos for kw in [
                "dành cho", "cho bé", "cho trẻ",
                "dùng cho", "sử dụng cho",
                "đối tượng sử dụng",
                "phù hợp cho", "phù hợp với",
                "độ tuổi phù hợp", "độ tuổi sử dụng",
                "trong độ tuổi",
                "giai đoạn từ", "giai đoạn dành cho",
                "khuyên dùng cho",
            ]):
                return

        # 4) POS/NEG context tổng quát
        if not has_pos_context(ctx_pos) or has_neg_context(ctx_neg):
            return

        matches.append(match_text)
        used_spans.append((start_idx, end_idx))

    # 0) Range "3M-5T"
    for m in AGE_MT_RANGE_PATTERN.finditer(text):
        handle_match(m)

    # 1) Range "0-6 tháng ... đến ... 3 tuổi"
    for m in CROSS_MONTH_RANGE_TO_YEAR_PATTERN.finditer(text):
        handle_match(m)

    # 2) Range newborn "sơ sinh ... đến ... 3 tuổi"
    for m in NEWBORN_RANGE_PATTERN.finditer(text):
        handle_match(m)

    # 3) Newborn phrase "trẻ sơ sinh trở lên", "từ trẻ sơ sinh"
    for m in NEWBORN_PHRASE_PATTERN.finditer(text):
        handle_match(m)

    # 4) Range 2 đơn vị: '6 tháng đến 5 tuổi', 'từ 1 tuần - 1 tuổi'
    for m in RANGE_BOTH_UNITS_PATTERN.finditer(text):
        handle_match(m)

    # 5) Range 1 đơn vị: '0-12 tháng', 'từ 2-6 tuổi', 'Từ 1-36 tháng tuổi'
    for m in RANGE_ONE_UNIT_PATTERN.finditer(text):
        handle_match(m)

    # 6) "độ tuổi từ 2 đến 6"
    for m in AGE_WORD_RANGE_PATTERN.finditer(text):
        handle_match(m)

    # 7) Single: 'dưới 12 tháng', 'từ 6 tháng tuổi', 'từ 1 ngày tuổi',
    #            'trên 6 tuổi', 'không quá 3 tuổi', '4 tháng tuổi'
    for m in SINGLE_PATTERN.finditer(text):
        handle_match(m)

    return matches


In [98]:
# =========================================
# 8. CHUẨN HOÁ VỀ CANONICAL (THÁNG / NGÀY)
# =========================================

def _to_months(num_str: str, unit: str) -> int:
    """Chuyển '1', '1,5', '2.5' -> số THÁNG theo unit."""
    val = float(num_str.replace(",", "."))
    unit = unit.lower()

    if unit in MONTH_WORDS:
        return int(round(val))
    if unit in YEAR_WORDS:
        return int(round(val * 12))
    if unit in WEEK_WORDS:
        m = int(round(val * 7.0 / 30.0))
        return max(0, m)

    raise ValueError(f"Unit không hợp lệ cho _to_months: {unit}")


def normalize_age_phrase(raw: str) -> Optional[str]:
    """
    Chuẩn hóa 1 cụm tuổi/tháng/ngày về canonical:

      - 'từ 3 tháng tuổi'           -> '3M+'
      - 'từ 2-6 tuổi'               -> '24-72M'
      - 'trong độ tuổi từ 1 đến 5'  -> '12-60M'
      - '0 - 48 tháng tuổi'         -> '0-48M'
      - '0-6 tháng'                 -> '0-6M'
      - 'sơ sinh ... đến 1 tuổi'    -> '0-12M'
      - 'trẻ sơ sinh trở lên'       -> '0M+'
      - 'từ 1 ngày tuổi'            -> '1D+'
      - 'dưới 3 tuổi'               -> '0-36M'
      - 'không quá 3 tuổi'          -> '0-36M'
      - (KHÔNG còn map single '4 tháng tuổi' / '2 tuổi' nếu không có từ so sánh)
    """
    if not raw:
        return None

    s2 = raw.strip().lower()
    s2 = s2.replace("tháng tuổi", " tháng ")
    s2 = s2.replace("thang toi", " thang ")
    s2 = re.sub(r"\s+", " ", s2)

    # 0) '3m-5t'
    m_mt = re.search(
        r"(\d{1,2})\s*m\s*(?:-|–|—|đến|to)\s*(\d{1,2})\s*t",
        s2,
    )
    if m_mt:
        a_str, b_str = m_mt.group(1), m_mt.group(2)
        start_m = _to_months(a_str, "tháng")
        end_m   = _to_months(b_str, "tuổi")
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{int(start_m)}-{int(end_m)}M"

    # 1) RANGE 2 ĐƠN VỊ: '6 tháng đến 5 tuổi', 'từ 1 tháng đến 2 tuổi'
    m_both = re.search(
        r"(?:từ\s*)?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
        r"(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_both:
        a_str, u1, b_str, u2 = m_both.group(1), m_both.group(2), m_both.group(3), m_both.group(4)
        start_m = _to_months(a_str, u1)
        end_m   = _to_months(b_str, u2)
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{int(start_m)}-{int(end_m)}M"

    # 2) RANGE 1 ĐƠN VỊ: '1-3 tuổi', 'từ 1-36 tháng'
    m_range_unit = re.search(
        r"(?:từ\s*)?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_range_unit:
        a_str = m_range_unit.group(1)
        b_str = m_range_unit.group(2)
        unit  = m_range_unit.group(3)
        start_m = _to_months(a_str, unit)
        end_m   = _to_months(b_str, unit)
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{int(start_m)}-{int(end_m)}M"

    # 3) 'độ tuổi từ 2 đến 6' (MẶC ĐỊNH NĂM, nếu KHÔNG chứa từ 'tháng')
    m_ageword = re.search(
        r"độ\s*tuổi[^0-9]{0,40}"
        r"(?:từ\s*)?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)",
        s2,
    )
    if m_ageword:
        sub = s2[m_ageword.start():m_ageword.end()]
        if not any(w in sub for w in ["tháng", "thang", "month", "months"]):
            a_str = m_ageword.group(1)
            b_str = m_ageword.group(2)
            start_m = _to_months(a_str, "tuổi")
            end_m   = _to_months(b_str, "tuổi")
            if start_m > end_m:
                start_m, end_m = end_m, start_m
            return f"{int(start_m)}-{int(end_m)}M"

    # 4) XỬ LÝ CÁC CASE CÓ 'SƠ SINH'
    if "sơ sinh" in s2:
        # 4.1. 'từ X ... đến Y ...' trong câu có 'sơ sinh' -> X-Y
        m_nb_range_xy = re.search(
            r"từ\s+(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years|y)\s*"
            r"(?:-|–|—|đến|to)\s*"
            r"(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years|y)",
            s2,
        )
        if m_nb_range_xy:
            a_str, u1, b_str, u2 = (
                m_nb_range_xy.group(1),
                m_nb_range_xy.group(2),
                m_nb_range_xy.group(3),
                m_nb_range_xy.group(4),
            )
            start_m = _to_months(a_str, u1)
            end_m   = _to_months(b_str, u2)
            if start_m > end_m:
                start_m, end_m = end_m, start_m
            return f"{int(start_m)}-{int(end_m)}M"

        # 4.2. 'sơ sinh ... đến ... N tháng/tuổi' -> 0-NM
        m_nb_range = re.search(
            r"(?:từ\s+giai\s*đoạn\s+)?sơ\s*sinh"
            r".{0,30}?(?:đến|tới|to|-).{0,30}?"
            r"(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years|y)",
            s2,
        )
        if m_nb_range:
            n_str = m_nb_range.group(1)
            unit  = m_nb_range.group(2)
            end_m = _to_months(n_str, unit)
            if end_m <= 0:
                return None
            return f"0-{int(end_m)}M"

        # 4.3. 'từ sơ sinh' / 'từ trẻ sơ sinh' -> 0M+
        m_nb_from0 = re.search(
            r"từ\s+(?:trẻ\s+)?sơ\s*sinh\b(?:\s*(trở lên|\+))?",
            s2,
        )
        if m_nb_from0:
            return "0M+"

        # 4.4. 'trẻ sơ sinh trở lên' -> 0M+
        m_nb_plus = re.search(
            r"(?:trẻ\s+)?sơ\s*sinh\s*(trở lên|\+)",
            s2,
        )
        if m_nb_plus:
            return "0M+"

        # 4.5. 'sơ sinh' / 'trẻ sơ sinh' ĐƠN LẺ -> bỏ (không fill)
        return None

    # 5) 'dưới / under / không quá N ...' -> '0-NM' hoặc '0-ND'
    m_under = re.search(
        r"(dưới|under|<|không quá|khong qua)\s*(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks|ngày tuổi|ngày|ngay|day|days)",
        s2,
    )
    if m_under:
        n_str = m_under.group(2)
        unit  = m_under.group(3).lower()
        if unit in DAY_WORDS:
            val = float(n_str.replace(",", "."))
            end_d = int(round(val))
            if end_d <= 0:
                return None
            return f"0-{end_d}D"
        else:
            end_m = _to_months(n_str, unit)
            if end_m <= 0:
                return None
            return f"0-{int(end_m)}M"

    # 6) 'từ / trên / hơn / sau N ...' -> 'NM+' hoặc 'ND+'
    m_from = re.search(
        r"(từ|trên|hơn|sau|>=|over|more than)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks|ngày tuổi|ngày|ngay|day|days)"
        r"(?:\s*(trở lên|\+))?",
        s2,
    )
    if m_from:
        n_str = m_from.group(2)
        unit  = m_from.group(3).lower()
        if unit in DAY_WORDS:
            val = float(n_str.replace(",", "."))
            start_d = int(round(val))
            return f"{start_d}D+"
        else:
            start_m = _to_months(n_str, unit)
            return f"{int(start_m)}M+"

    # 7) 'N ... trở lên' -> 'NM+' hoặc 'ND+'
    m_suffix_plus = re.search(
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks|ngày tuổi|ngày|ngay|day|days)\s*"
        r"(trở lên|\+)",
        s2,
    )
    if m_suffix_plus:
        n_str = m_suffix_plus.group(1)
        unit  = m_suffix_plus.group(2).lower()
        if unit in DAY_WORDS:
            val = float(n_str.replace(",", "."))
            start_d = int(round(val))
            return f"{start_d}D+"
        else:
            start_m = _to_months(n_str, unit)
            return f"{int(start_m)}M+"

    # 8) KHÔNG BẮT SINGLE '4 tháng tuổi' / '2 tuổi' nếu không có từ so sánh
    # => trả None
    return None


In [99]:
def normalize_age_phrase_list(raw_list: Optional[List[str]]) -> List[str]:
    """
    Chuẩn hoá list cụm tuổi và GIỮ TẤT CẢ các khoảng hợp lệ
    (chỉ loại bỏ trùng lặp exact-string, KHÔNG xoá khoảng con như 0-6M khi có 0-12M).
    """
    if raw_list is None:
        return []

    uniq: List[str] = []
    for raw in raw_list:
        norm = normalize_age_phrase(raw)
        if norm is not None and norm not in uniq:
            uniq.append(norm)

    return uniq


In [100]:
# df_unknown_desc đã được bạn tạo từ trước, gồm:
# item_id, description, description_new, age_group_final
# với điều kiện:
#   - age_group_final == "Không xác định"
#   - ít nhất 1 trong 2 description có thông tin

df_unknown = (
    df_unknown_desc
    .with_columns(
        pl.concat_str(
            [
                pl.col("description").fill_null(""),
                pl.col("description_new").fill_null(""),
            ],
            separator=" "
        ).alias("desc_all")
    )
    .with_columns(
        pl.col("desc_all").map_elements(
            extract_age_phrases,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_phrases_raw")
    )
    .with_columns(
        pl.col("age_phrases_raw").map_elements(
            normalize_age_phrase_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_ranges_norm")
    )
)

# Lấy tất cả canonical ranges đã detect
target_norm_list = (
    df_unknown
    .select(pl.col("age_ranges_norm"))
    .explode("age_ranges_norm")
    .filter(pl.col("age_ranges_norm").is_not_null())
    .unique()
    .to_series()
    .to_list()
)

print("Số lượng target_norm:", len(target_norm_list))
print(target_norm_list)

# Xuất ví dụ để review
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(50)

def wrap175(text):
    if text is None:
        return ""
    return textwrap.fill(str(text), width=175)

with open("output_age_norm.txt", "w", encoding="utf-8") as f:

    for tg in target_norm_list:
        f.write(f"\n=== VÍ DỤ CHO TARGET_NORM = {tg} ===\n\n")
        target_norm = tg

        examples = (
            df_unknown
            .filter(pl.col("age_ranges_norm").list.contains(target_norm))
            .select([
                "item_id",
                "age_group_final",
                "age_phrases_raw",
                "age_ranges_norm",
                "description",
                "description_new",
            ])
            .head(20)
        )

        for row in examples.iter_rows(named=True):
            f.write("======================================\n")
            f.write(f"item_id: {row['item_id']}\n")
            f.write(f"age_group_final: {row['age_group_final']}\n")
            f.write(f"age_phrases_raw: {row['age_phrases_raw']}\n")
            f.write(f"age_ranges_norm: {row['age_ranges_norm']}\n\n")

            f.write("description:\n")
            f.write(wrap175(row["description"]) + "\n\n")

            f.write("description_new:\n")
            f.write(wrap175(row["description_new"]) + "\n\n")

print("Đã xuất file: output_age_norm.txt")


Số lượng target_norm: 42
['228M+', '2M+', '0-60M', '156M+', '12M+', '7-24M', '0-12M', '216M+', '120M+', '6M+', '36-216M', '36M+', '12-36M', '4M+', '0-120M', '3M+', '0-144M', '0-36M', '0-24M', '12-72M', '3-24M', '12-60M', '0-3M', '6-60M', '8M+', '0-6M', '36-96M', '10M+', '24-72M', '9-24M', '72M+', '84-144M', '6-30M', '24M+', '0-48M', '48M+', '3-6M', '0M+', '0-72M', '60M+', '6-12M', '9-12M']
Đã xuất file: output_age_norm.txt


Fill các giá trị từ target_norm

In [101]:
import polars as pl

# df_unknown hiện đang có:
# - item_id
# - age_ranges_norm: List[str] (canonical, ví dụ: ["0-12M", "12-36M", "60M+"])

# 1) explode để mỗi dòng là 1 (item_id, age_range)
df_item_norm = (
    df_unknown
    .select(["item_id", "age_ranges_norm"])
    .explode("age_ranges_norm")
    .filter(pl.col("age_ranges_norm").is_not_null())
)

# 2) định nghĩa hàm chuyển "XM+" -> "Từ XM"
def convert_plus_list(lst: list[str]) -> list[str]:
    out = []
    for s in lst:
        if s.endswith("M+"):
            num = s[:-2]           # "60M+" -> "60"
            out.append(f"Từ {num}M")
        else:
            out.append(s)
    return out

# 3) group_by theo item_id, unique + sort, rồi apply convert_plus_list
df_item_age_dict = (
    df_item_norm
    .group_by("item_id")
    .agg(
        pl.col("age_ranges_norm").unique()  # list các canonical khác nhau
    )
    .with_columns(
        pl.col("age_ranges_norm").map_elements(
            convert_plus_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_dict_list")
    )
    # tạo thêm bản string dạng ["0-12M","Từ 12M"] để fill vào age_group_final
    .with_columns(
        pl.col("age_dict_list").map_elements(
            lambda lst: "[" + ", ".join(f'"{x}"' for x in lst) + "]",
            return_dtype=pl.Utf8
        ).alias("age_dict_str")
    )
)

# df_item_age_dict hiện có:
# - item_id
# - age_ranges_norm (list canonical gốc)
# - age_dict_list (list đã đổi M+ -> "Từ XM")
# - age_dict_str  (string để fill vào age_group_final)


In [102]:
# Tạo dictionary dùng cho việc debug / dùng ngoài Polars
item_to_age_dict = {
    row["item_id"]: row["age_dict_list"]
    for row in df_item_age_dict.select(["item_id", "age_dict_list"]).to_dicts()
}

# Giờ item_to_age_dict[item_id] sẽ là list, ví dụ:
# ["0-12M", "12-36M", "Từ 60M"]


In [103]:
# Join mapping theo item_id, chỉ fill cho các dòng "Không xác định"

df_age_filled = (
    df_age
    .join(
        df_item_age_dict.select(["item_id", "age_dict_str"]),
        on="item_id",
        how="left"
    )
    .with_columns(
        pl.when(
            (pl.col("age_group_final") == "Không xác định")
            & pl.col("age_dict_str").is_not_null()
        )
        .then(pl.col("age_dict_str"))         # dùng dictionary dạng ["...","..."]
        .otherwise(pl.col("age_group_final"))
        .alias("age_group_final_filled")
    )
    .drop("age_group_final", "age_dict_str")
    .rename({"age_group_final_filled": "age_group_final"})
)

# df_age_filled là dataframe cuối cùng sau khi fill.


In [104]:
import polars as pl

# df_unknown: item_id, age_ranges_norm

df_item_norm = (
    df_unknown
    .select(["item_id", "age_ranges_norm"])
    .explode("age_ranges_norm")
    .filter(pl.col("age_ranges_norm").is_not_null())
)

def convert_plus_list(lst):
    # lst có thể là list hoặc Series
    if isinstance(lst, pl.Series):
        lst = lst.to_list()

    out = []
    for s in lst:
        if not isinstance(s, str):
            out.append(s)
            continue

        s_clean = s.strip()

        # CASE 1: dạng "XM+"
        if s_clean.endswith("M+"):
            num = s_clean[:-2]
            out.append(f"Từ {num}M")
            continue

        # CASE 2: dạng chữ "trên 6M" / "Trên 6M" / "hơn 12M" / "Hơn 12M"
        m = re.match(r"(?i)^(trên|hơn)\s*(\d+)\s*m$", s_clean)
        if m:
            num = m.group(2)
            out.append(f"Từ {num}M")
            continue

        # giữ nguyên các dạng khác ("0-6M", "6-12M", ...)
        out.append(s_clean)

    return out

def build_age_fill_str(lst):
    if isinstance(lst, pl.Series):
        lst = lst.to_list()
    if lst is None or len(lst) == 0:
        return None
    if len(lst) == 1:
        # trả về string đơn
        return lst[0]
    # nhiều giá trị → JSON-like list với dấu "
    return "[" + ", ".join(f"\"{x}\"" for x in lst) + "]"

df_item_age_dict = (
    df_item_norm
    .group_by("item_id")
    .agg(pl.col("age_ranges_norm").unique().sort())
    .with_columns(
        pl.col("age_ranges_norm").map_elements(
            convert_plus_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_dict_list")
    )
    .with_columns(
        pl.col("age_dict_list").map_elements(
            build_age_fill_str,
            return_dtype=pl.Utf8
        ).alias("age_fill_str")
    )
)

In [105]:
# Thống kê trước khi fill
total_rows = df_age.height
unknown_before = df_age.filter(pl.col("age_group_final") == "Không xác định").height
ratio_before = unknown_before / total_rows * 100

print("=== TRƯỚC KHI FILL ===")
print(f"Số dòng 'Không xác định': {unknown_before}")
print(f"Tỉ lệ: {ratio_before:.2f}%")
print()


# Fill
df_age_filled = (
    df_age
    .with_columns(
        pl.col("age_group_final").alias("age_group_final_before")
    )
    .join(
        df_item_age_dict.select(["item_id", "age_fill_str"]),
        on="item_id",
        how="left"
    )
    .with_columns(
        pl.when(
            (pl.col("age_group_final_before") == "Không xác định")
            & pl.col("age_fill_str").is_not_null()
        )
        .then(pl.col("age_fill_str"))
        .otherwise(pl.col("age_group_final_before"))
        .alias("age_group_final")
    )
    .drop("age_fill_str")
)

# KHÔNG đổi dấu nháy nữa. Không .str.replace() gì thêm.


# Thống kê sau fill
unknown_after = df_age_filled.filter(pl.col("age_group_final") == "Không xác định").height
ratio_after = unknown_after / total_rows * 100

print("=== SAU KHI FILL ===")
print(f"Số dòng 'Không xác định': {unknown_after}")
print(f"Tỉ lệ: {ratio_after:.2f}%")
print()


# Lấy ra các dòng đã fill
df_filled_rows = (
    df_age_filled
    .filter(
        (pl.col("age_group_final_before") == "Không xác định")
        & (pl.col("age_group_final") != "Không xác định")
    )
    .select([
        "item_id",
        "description",
        "description_new",
        "age_group_final_before",
        "age_group_final",
    ])
)

pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_rows(100)

print("=== CÁC DÒNG ĐÃ FILL ===")
print(df_filled_rows)

=== TRƯỚC KHI FILL ===
Số dòng 'Không xác định': 10362
Tỉ lệ: 37.92%

=== SAU KHI FILL ===
Số dòng 'Không xác định': 10085
Tỉ lệ: 36.91%

=== CÁC DÒNG ĐÃ FILL ===
shape: (277, 5)
┌───────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│ item_id       ┆ description        ┆ description_new    ┆ age_group_final_be ┆ age_group_final   │
│ ---           ┆ ---                ┆ ---                ┆ fore               ┆ ---               │
│ str           ┆ str                ┆ str                ┆ ---                ┆ str               │
│               ┆                    ┆                    ┆ str                ┆                   │
╞═══════════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════════╡
│ 0006040000078 ┆ Bình muỗng ăn dặm  ┆ Không xác định     ┆ Không xác định     ┆ Từ 6M             │
│               ┆ silicone mềm       ┆                    ┆                    ┆                   │
│            

In [106]:
split_and_save_parquet(df_age_filled, 1, "./preprocessed-dataset")

Đã lưu file: ./preprocessed-dataset/sale_pers.item_chunk_0.parquet
